# Handwritten Digit Clustering

Test how well unsupervised clusters recover digit structure from raw pixel intensities.

**Portfolio category:** Clustering

**Data mode:** Built-in dataset

This notebook keeps labels out of fitting wherever labels exist, uses deterministic seeds,
reports unsupervised-specific diagnostics, and avoids hard-coded results.

## 1. Project setup

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

## 2. Load handwritten digit pixels

In [ ]:
digits = load_digits()
X_raw = digits.data
hidden_labels = digits.target
print("Samples:", X_raw.shape[0], "Features:", X_raw.shape[1])

## 3. Pixel quality and examples

In [ ]:
print("Missing values:", int(np.isnan(X_raw).sum()))
fig, axes = plt.subplots(2, 6, figsize=(10, 4))
for axis, image in zip(axes.flat, digits.images[:12]):
    axis.imshow(image, cmap="gray_r")
    axis.axis("off")
plt.tight_layout()

## 4. Scale and reduce dimensions

In [ ]:
X_scaled = StandardScaler().fit_transform(X_raw)
pca = PCA(n_components=0.90, random_state=RANDOM_STATE)
X = pca.fit_transform(X_scaled)
print("Components retaining 90% variance:", X.shape[1])

## 5. Cluster the digits

In [ ]:
model = KMeans(n_clusters=10, n_init=50, random_state=RANDOM_STATE)
clusters = model.fit_predict(X)

## 6. Evaluate structure; labels were not used for fitting

In [ ]:
contingency = pd.crosstab(pd.Series(clusters, name="cluster"), pd.Series(hidden_labels, name="digit"))
purity = contingency.max(axis=1).sum() / contingency.to_numpy().sum()
display(pd.Series({
    "silhouette": silhouette_score(X, clusters),
    "adjusted_rand_index": adjusted_rand_score(hidden_labels, clusters),
    "normalized_mutual_information": normalized_mutual_info_score(hidden_labels, clusters),
    "cluster_purity": purity,
}).to_frame("value"))

## 7. Cluster-to-digit diagnostic

In [ ]:
sns.heatmap(contingency, cmap="Blues")
plt.title("Cluster composition by hidden digit label")
plt.tight_layout()

## 8. PCA projection

In [ ]:
projection = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_scaled)
sns.scatterplot(x=projection[:, 0], y=projection[:, 1], hue=clusters, palette="tab10", s=20, legend=False)
plt.title("Digit clusters in two-dimensional PCA space")
plt.tight_layout()

## 9. Key findings

Visual similarity does not map perfectly to digit identity; confusion between similarly shaped digits is expected.

## 10. Interpretation and responsible use

Treat the output as exploratory evidence, not ground truth. For handwritten digit clustering,
validate stability on newer data, inspect edge cases, and review domain risks before
turning clusters, rankings or anomaly scores into decisions.

## 11. Next steps

- Replace demonstration data with a versioned, licensed dataset.
- Track data quality, drift and stability across repeated runs.
- Add domain-specific review before deployment.
- Package inference only after reproducibility and privacy checks pass.

All numeric results are generated at execution time; none are hard-coded.